In [ ]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import os

# ==============================
# CONFIG
# ==============================
OUTPUT_FILE = "synthetic_fast.parquet"
BATCH_SIZE = 2000
DAYS = 7

# ==============================
# 1. SMALL TILE GRID (CRITICAL FIX)
# ==============================
print("\n[STEP 1] Downloading Mumbai using SMALL tiles...\n")

north_vals = np.arange(19.30, 19.00, -0.05)
south_vals = north_vals - 0.05

east_vals = np.arange(72.80, 73.10, 0.05)
west_vals = east_vals - 0.05

graphs = []
tile_count = 0

for n, s in zip(north_vals, south_vals):
    for e, w in zip(east_vals, west_vals):

        bbox = (n, s, e, w)
        print(f"[TILE {tile_count}] {bbox}")

        try:
            g = ox.graph_from_bbox(bbox, network_type="drive")
            graphs.append(g)
            print("   ✅ Done")
        except Exception as ex:
            print("   ❌ Failed:", ex)

        tile_count += 1

print("\n[STEP 1 DONE] Merging graphs...\n")
G = nx.compose_all(graphs)

nodes, edges = ox.graph_to_gdfs(G)
edges = edges[["u", "v", "length", "highway", "maxspeed", "lanes"]].copy()
edges.reset_index(drop=True, inplace=True)

print(f"✅ Total edges: {len(edges)}\n")

# ==============================
# 2. CLEAN FEATURES
# ==============================
print("[STEP 2] Cleaning features...\n")

def parse_speed(x):
    if isinstance(x, list):
        x = x[0]
    try:
        return float(str(x).replace(" km/h", ""))
    except:
        return 40.0

edges["maxspeed"] = edges["maxspeed"].apply(parse_speed)

edges["lanes"] = edges["lanes"].apply(
    lambda x: int(x[0]) if isinstance(x, list) else (int(x) if x else 2)
)

edges["free_flow_speed"] = edges["maxspeed"]

road_weight = {
    "motorway": 1.0,
    "trunk": 0.9,
    "primary": 0.8,
    "secondary": 0.7,
    "tertiary": 0.6,
    "residential": 0.5
}

edges["road_weight"] = edges["highway"].apply(
    lambda x: road_weight.get(x[0] if isinstance(x, list) else x, 0.5)
)

edge_features = edges[[
    "u", "v", "length", "free_flow_speed", "road_weight"
]].to_numpy()

print("✅ Features ready\n")

# ==============================
# 3. TIME VECTOR
# ==============================
print("[STEP 3] Generating time...\n")

timestamps = pd.date_range(
    start="2024-01-01",
    periods=24 * DAYS,
    freq="1H"
)

hours = timestamps.hour.values
days = timestamps.weekday.values
T = len(timestamps)

print(f"✅ Time steps: {T}\n")

# ==============================
# 4. TIME FACTORS
# ==============================
print("[STEP 4] Computing time patterns...\n")

peak = np.where(((hours >= 8) & (hours <= 11)) | ((hours >= 17) & (hours <= 21)), 0.6, 1.0)
weekend = np.where(days >= 5, 0.8, 1.0)
time_factor = (peak * weekend)[None, :]

print("✅ Time factors ready\n")

# ==============================
# 5. BATCH PROCESSING (VECTOR)
# ==============================
print("[STEP 5] Processing batches...\n")

if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

for i in range(0, len(edge_features), BATCH_SIZE):

    print(f"\n🚀 Batch {i} → {min(i+BATCH_SIZE, len(edge_features))}")

    batch = edge_features[i:i+BATCH_SIZE]

    u = batch[:, 0]
    v = batch[:, 1]
    length = batch[:, 2][:, None]
    free_speed = batch[:, 3][:, None]
    road_w = batch[:, 4][:, None]

    N = len(batch)
    print(f"   Edges: {N}")

    # Random factors
    rain = np.random.choice([0, 1], size=(N, T), p=[0.8, 0.2])
    incident = np.random.choice([0, 1], size=(N, T), p=[0.95, 0.05])

    weather_factor = np.where(rain == 1, 0.7, 1.0)
    incident_factor = np.where(incident == 1, 0.5, 1.0)

    signal_factor = np.where(road_w < 0.7, 0.85, 1.0)
    noise = np.random.normal(1.0, 0.1, size=(N, T))

    # Speed
    speed = (
        free_speed
        * time_factor
        * road_w
        * weather_factor
        * incident_factor
        * signal_factor
        * noise
    )

    speed = np.clip(speed, 5, free_speed)

    congestion = 1 - (speed / free_speed)
    travel_time = length / (speed * 1000 / 3600)

    print("   ✅ Computed metrics")

    # Flatten
    df = pd.DataFrame({
        "u": np.repeat(u, T),
        "v": np.repeat(v, T),
        "timestamp": np.tile(timestamps, N),
        "hour": np.tile(hours, N),
        "day_of_week": np.tile(days, N),
        "current_speed": speed.flatten(),
        "free_flow_speed": np.repeat(free_speed.flatten(), T),
        "congestion": congestion.flatten(),
        "travel_time": travel_time.flatten(),
        "rain": rain.flatten(),
        "incident": incident.flatten()
    })

    print(f"   📦 Shape: {df.shape}")

    # Save
    if i == 0:
        df.to_parquet(OUTPUT_FILE, compression="snappy")
    else:
        df.to_parquet(OUTPUT_FILE, compression="snappy", append=True)

    print("   💾 Saved")

print("\n🎉 DONE → synthetic_fast.parquet\n")


[STEP 1] Downloading Mumbai using SMALL tiles...

[TILE 0] (np.float64(19.3), np.float64(19.25), np.float64(72.8), np.float64(72.75))


d:\Urban-Traffic-and-Parking-Analysis-using-LSTM-Autoencoders-and-Reinforcement-Learning\venv\Lib\site-packages\osmnx\_overpass.py:271: UserWarning: This area is 8,869 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [ ]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import os
import time

# ==============================
# CONFIG
# ==============================
OUTPUT_FILE = "synthetic_fast.parquet"
BATCH_SIZE = 2000
DAYS = 7
DIST = 3000

# ==============================
# 1. DOWNLOAD USING POINT GRID
# ==============================
print("\n[STEP 1] Downloading Mumbai using POINT grid...\n")

lats = np.arange(19.00, 19.30, 0.05)
lons = np.arange(72.80, 73.10, 0.05)

graphs = []
tile_count = 0

for lat in lats:
    for lon in lons:

        print(f"[TILE {tile_count}] Center: ({lat:.3f}, {lon:.3f})")

        success = False

        for attempt in range(3):
            try:
                g = ox.graph_from_point(
                    (lat, lon),
                    dist=DIST,
                    network_type="drive"
                )
                graphs.append(g)
                print("   ✅ Done")
                success = True
                break
            except Exception:
                print(f"   ❌ Retry {attempt+1}")
                time.sleep(2)

        if not success:
            print("   ❌ Skipped tile")

        tile_count += 1

print("\n[STEP 1 DONE] Merging graphs...\n")

if len(graphs) == 0:
    raise Exception("No graphs downloaded.")

G = nx.compose_all(graphs)

nodes, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index()
print(edges.columns)
# Safe column selection
required_cols = ["u", "v", "length", "highway", "maxspeed", "lanes"]
for col in required_cols:
    if col not in edges.columns:
        edges[col] = np.nan

edges = edges[required_cols].copy()

print(f"✅ Total edges: {len(edges)}\n")

# ==============================
# 2. ROBUST CLEANING (FINAL FIX)
# ==============================

print("[STEP 2] Cleaning features...\n")

def normalize(x):
    if isinstance(x, (list, np.ndarray)):
        if len(x) == 0:
            return None
        x = x[0]
    return x

def parse_speed(x):
    x = normalize(x)
    if x is None or pd.isna(x):
        return 40.0
    try:
        return float(str(x).replace(" km/h", ""))
    except:
        return 40.0

def parse_lanes(x):
    x = normalize(x)
    if x is None or pd.isna(x):
        return 2
    try:
        return int(float(x))
    except:
        return 2

edges["maxspeed"] = edges["maxspeed"].apply(parse_speed)
edges["lanes"] = edges["lanes"].apply(parse_lanes)

edges["free_flow_speed"] = edges["maxspeed"]

road_weight = {
    "motorway": 1.0,
    "trunk": 0.9,
    "primary": 0.8,
    "secondary": 0.7,
    "tertiary": 0.6,
    "residential": 0.5
}

def get_weight(x):
    if isinstance(x, (list, np.ndarray)):
        if len(x) == 0:
            return 0.5
        x = x[0]
    return road_weight.get(x, 0.5)

edges["road_weight"] = edges["highway"].apply(get_weight)

edge_features = edges[[
    "u", "v", "length", "free_flow_speed", "road_weight"
]].to_numpy()

print("✅ Feature cleaning done\n")

# ==============================
# 3. TIME VECTOR
# ==============================

print("[STEP 3] Generating time...\n")

timestamps = pd.date_range(
    start="2024-01-01",
    periods=24 * DAYS,
    freq="1h"
)

hours = timestamps.hour.values
days = timestamps.weekday.values
T = len(timestamps)

peak = np.where(((hours >= 8) & (hours <= 11)) | ((hours >= 17) & (hours <= 21)), 0.6, 1.0)
weekend = np.where(days >= 5, 0.8, 1.0)
time_factor = (peak * weekend)[None, :]

print(f"✅ Time steps: {T}\n")

# ==============================
# 4. GENERATE DATA
# ==============================

print("[STEP 4] Generating traffic data...\n")

OUTPUT_CSV = "synthetic_fast.csv"

if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

for i in range(0, len(edge_features), BATCH_SIZE):

    print(f"\n🚀 Batch {i} → {min(i+BATCH_SIZE, len(edge_features))}")

    batch = edge_features[i:i+BATCH_SIZE]

    u = batch[:, 0]
    v = batch[:, 1]
    length = batch[:, 2][:, None]
    free_speed = batch[:, 3][:, None]
    road_w = batch[:, 4][:, None]

    N = len(batch)

    rain = np.random.choice([0, 1], size=(N, T), p=[0.8, 0.2])
    incident = np.random.choice([0, 1], size=(N, T), p=[0.95, 0.05])

    weather_factor = np.where(rain == 1, 0.7, 1.0)
    incident_factor = np.where(incident == 1, 0.5, 1.0)

    signal_factor = np.where(road_w < 0.7, 0.85, 1.0)
    noise = np.random.normal(1.0, 0.1, size=(N, T))

    speed = (
        free_speed
        * time_factor
        * road_w
        * weather_factor
        * incident_factor
        * signal_factor
        * noise
    )

    speed = np.clip(speed, 5, free_speed)

    congestion = 1 - (speed / free_speed)
    travel_time = length / (speed * 1000 / 3600)

    df = pd.DataFrame({
        "u": np.repeat(u, T),
        "v": np.repeat(v, T),
        "timestamp": np.tile(timestamps, N),
        "hour": np.tile(hours, N),
        "day_of_week": np.tile(days, N),
        "current_speed": speed.flatten(),
        "free_flow_speed": np.repeat(free_speed.flatten(), T),
        "congestion": congestion.flatten(),
        "travel_time": travel_time.flatten(),
        "rain": rain.flatten(),
        "incident": incident.flatten()
    })

    # ✅ SINGLE CLEAN WRITE
    df.to_csv(
        OUTPUT_CSV,
        mode='a',
        header=(i == 0),   # only first batch writes header
        index=False,
        float_format="%.3f"   # faster + smaller file
    )

    print("   💾 Saved")

print("\n🎉 DONE → synthetic_fast.csv\n")


[STEP 1] Downloading Mumbai using POINT grid...

[TILE 0] Center: (19.000, 72.800)
   ✅ Done
[TILE 1] Center: (19.000, 72.850)
   ✅ Done
[TILE 2] Center: (19.000, 72.900)
   ✅ Done
[TILE 3] Center: (19.000, 72.950)
   ❌ Retry 1
   ❌ Retry 2
   ❌ Retry 3
   ❌ Skipped tile
[TILE 4] Center: (19.000, 73.000)
   ✅ Done
[TILE 5] Center: (19.000, 73.050)
   ✅ Done
[TILE 6] Center: (19.050, 72.800)
   ✅ Done
[TILE 7] Center: (19.050, 72.850)
   ✅ Done
[TILE 8] Center: (19.050, 72.900)
   ✅ Done
[TILE 9] Center: (19.050, 72.950)
   ✅ Done
[TILE 10] Center: (19.050, 73.000)
   ✅ Done
[TILE 11] Center: (19.050, 73.050)
   ✅ Done
[TILE 12] Center: (19.100, 72.800)
   ✅ Done
[TILE 13] Center: (19.100, 72.850)
   ✅ Done
[TILE 14] Center: (19.100, 72.900)
   ✅ Done
[TILE 15] Center: (19.100, 72.950)
   ✅ Done
[TILE 16] Center: (19.100, 73.000)
   ✅ Done
[TILE 17] Center: (19.100, 73.050)
   ✅ Done
[TILE 18] Center: (19.150, 72.800)
   ✅ Done
[TILE 19] Center: (19.150, 72.850)
   ✅ Done
[TILE 20] Cen

In [8]:
data=pd.read_csv('synthetic_fast.csv')

In [9]:
len(data)

16967664

In [14]:
data.head(5)

,u,v,timestamp,hour,day_of_week,current_speed,free_flow_speed,congestion,travel_time,rain,incident
0,245657555.0,245657631.0,2024-01-01 00:00:00,0,0,14.547,40.0,0.636,29.991,0,0
1,245657555.0,245657631.0,2024-01-01 01:00:00,1,0,19.137,40.0,0.522,22.798,0,0
2,245657555.0,245657631.0,2024-01-01 02:00:00,2,0,16.824,40.0,0.579,25.933,0,0
3,245657555.0,245657631.0,2024-01-01 03:00:00,3,0,17.280,40.0,0.568,25.247,0,0
4,245657555.0,245657631.0,2024-01-01 04:00:00,4,0,13.508,40.0,0.662,32.299,0,0


In [17]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import os
import time

# ==============================
# CONFIG
# ==============================
OUTPUT_CSV = "synthetic_full.csv"
BATCH_SIZE = 2000
DAYS = 7
DIST = 3000

# ==============================
# 1. DOWNLOAD GRAPH
# ==============================
print("\n[STEP 1] Downloading Mumbai...\n")

lats = np.arange(19.00, 19.30, 0.05)
lons = np.arange(72.80, 73.10, 0.05)

graphs = []
tile_count = 0

for lat in lats:
    for lon in lons:

        print(f"[TILE {tile_count}] ({lat:.3f}, {lon:.3f})")

        for attempt in range(3):
            try:
                g = ox.graph_from_point(
                    (lat, lon),
                    dist=DIST,
                    network_type="drive"
                )
                graphs.append(g)
                print("   ✅ Done")
                break
            except:
                print(f"   ❌ Retry {attempt+1}")
                time.sleep(2)

        tile_count += 1

print("\n[STEP 1 DONE] Merging...\n")

G = nx.compose_all(graphs)

# ==============================
# 2. EXTRACT FULL DATA
# ==============================

nodes, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index()

# Add node coordinates
nodes = nodes[["x", "y"]]  # lon, lat

edges["u_lon"] = edges["u"].map(nodes["x"])
edges["u_lat"] = edges["u"].map(nodes["y"])
edges["v_lon"] = edges["v"].map(nodes["x"])
edges["v_lat"] = edges["v"].map(nodes["y"])

# Keep ALL useful columns (no restriction)
print("Available edge columns:", edges.columns.tolist())

# ==============================
# 3. CLEAN IMPORTANT FEATURES
# ==============================

def normalize(x):
    if isinstance(x, (list, np.ndarray)):
        return x[0] if len(x) > 0 else None
    return x

def parse_speed(x):
    x = normalize(x)
    if x is None or pd.isna(x):
        return 40.0
    try:
        return float(str(x).replace(" km/h", ""))
    except:
        return 40.0

def parse_lanes(x):
    x = normalize(x)
    if x is None or pd.isna(x):
        return 2
    try:
        return int(float(x))
    except:
        return 2

edges["maxspeed"] = edges["maxspeed"].apply(parse_speed)
edges["lanes"] = edges["lanes"].apply(parse_lanes)

edges["free_flow_speed"] = edges["maxspeed"]

road_weight = {
    "motorway": 1.0,
    "trunk": 0.9,
    "primary": 0.8,
    "secondary": 0.7,
    "tertiary": 0.6,
    "residential": 0.5
}

edges["road_weight"] = edges["highway"].apply(
    lambda x: road_weight.get(normalize(x), 0.5)
)

# ==============================
# 4. TIME VECTOR
# ==============================

timestamps = pd.date_range(
    start="2024-01-01",
    periods=24 * DAYS,
    freq="1h"
)

hours = timestamps.hour.values
days = timestamps.weekday.values
T = len(timestamps)

peak = np.where(((hours >= 8) & (hours <= 11)) | ((hours >= 17) & (hours <= 21)), 0.6, 1.0)
weekend = np.where(days >= 5, 0.8, 1.0)
time_factor = (peak * weekend)[None, :]

print(f"Time steps: {T}")

# ==============================
# 5. GENERATE DATA
# ==============================

if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

edge_features = edges[[
    "u", "v", "length", "free_flow_speed", "road_weight",
    "u_lat", "u_lon", "v_lat", "v_lon"
]].to_numpy()

for i in range(0, len(edge_features), BATCH_SIZE):

    print(f"\n🚀 Batch {i} → {min(i+BATCH_SIZE, len(edge_features))}")

    batch = edge_features[i:i+BATCH_SIZE]

    u = batch[:, 0]
    v = batch[:, 1]
    length = batch[:, 2][:, None]
    free_speed = batch[:, 3][:, None]
    road_w = batch[:, 4][:, None]

    u_lat = batch[:, 5]
    u_lon = batch[:, 6]
    v_lat = batch[:, 7]
    v_lon = batch[:, 8]

    N = len(batch)

    rain = np.random.choice([0, 1], size=(N, T), p=[0.8, 0.2])
    incident = np.random.choice([0, 1], size=(N, T), p=[0.95, 0.05])

    weather_factor = np.where(rain == 1, 0.7, 1.0)
    incident_factor = np.where(incident == 1, 0.5, 1.0)

    signal_factor = np.where(road_w < 0.7, 0.85, 1.0)
    noise = np.random.normal(1.0, 0.1, size=(N, T))

    speed = (
        free_speed
        * time_factor
        * road_w
        * weather_factor
        * incident_factor
        * signal_factor
        * noise
    )

    speed = np.clip(speed, 5, free_speed)

    congestion = 1 - (speed / free_speed)
    travel_time = length / (speed * 1000 / 3600)

    df = pd.DataFrame({
        "u": np.repeat(u, T),
        "v": np.repeat(v, T),
        "u_lat": np.repeat(u_lat, T),
        "u_lon": np.repeat(u_lon, T),
        "v_lat": np.repeat(v_lat, T),
        "v_lon": np.repeat(v_lon, T),
        "length_m": np.repeat(length.flatten(), T),
        "timestamp": np.tile(timestamps, N),
        "hour": np.tile(hours, N),
        "day_of_week": np.tile(days, N),
        "current_speed": speed.flatten(),
        "free_flow_speed": np.repeat(free_speed.flatten(), T),
        "congestion": congestion.flatten(),
        "travel_time_sec": travel_time.flatten(),
        "rain": rain.flatten(),
        "incident": incident.flatten()
    })

    df.to_csv(
        OUTPUT_CSV,
        mode='a',
        header=(i == 0),
        index=False,
        float_format="%.3f"
    )

    print("   💾 Saved")

print("\n🎉 DONE → synthetic_full.csv\n")


[STEP 1] Downloading Mumbai...

[TILE 0] (19.000, 72.800)
   ✅ Done
[TILE 1] (19.000, 72.850)
   ✅ Done
[TILE 2] (19.000, 72.900)
   ✅ Done
[TILE 3] (19.000, 72.950)
   ❌ Retry 1
   ❌ Retry 2
   ❌ Retry 3
[TILE 4] (19.000, 73.000)
   ✅ Done
[TILE 5] (19.000, 73.050)
   ✅ Done
[TILE 6] (19.050, 72.800)
   ✅ Done
[TILE 7] (19.050, 72.850)
   ✅ Done
[TILE 8] (19.050, 72.900)
   ✅ Done
[TILE 9] (19.050, 72.950)
   ✅ Done
[TILE 10] (19.050, 73.000)
   ✅ Done
[TILE 11] (19.050, 73.050)
   ✅ Done
[TILE 12] (19.100, 72.800)
   ✅ Done
[TILE 13] (19.100, 72.850)
   ✅ Done
[TILE 14] (19.100, 72.900)
   ✅ Done
[TILE 15] (19.100, 72.950)
   ✅ Done
[TILE 16] (19.100, 73.000)
   ✅ Done
[TILE 17] (19.100, 73.050)
   ✅ Done
[TILE 18] (19.150, 72.800)
   ✅ Done
[TILE 19] (19.150, 72.850)
   ✅ Done
[TILE 20] (19.150, 72.900)
   ✅ Done
[TILE 21] (19.150, 72.950)
   ✅ Done
[TILE 22] (19.150, 73.000)
   ✅ Done
[TILE 23] (19.150, 73.050)
   ✅ Done
[TILE 24] (19.200, 72.800)
   ✅ Done
[TILE 25] (19.200, 72.8

In [20]:
data_2=pd.read_csv('synthetic_full.csv')

In [21]:
len(data_2)
# 16967664

16967664

In [22]:
data_2.head()

,u,v,u_lat,u_lon,v_lat,v_lon,length_m,timestamp,hour,day_of_week,current_speed,free_flow_speed,congestion,travel_time_sec,rain,incident
0,245657555.0,245657631.0,18.973,72.811,18.974,72.811,121.19,2024-01-01 00:00:00,0,0,18.636,40.0,0.534,23.411,0,0
1,245657555.0,245657631.0,18.973,72.811,18.974,72.811,121.19,2024-01-01 01:00:00,1,0,14.010,40.0,0.650,31.140,1,0
2,245657555.0,245657631.0,18.973,72.811,18.974,72.811,121.19,2024-01-01 02:00:00,2,0,17.446,40.0,0.564,25.008,0,0
3,245657555.0,245657631.0,18.973,72.811,18.974,72.811,121.19,2024-01-01 03:00:00,3,0,16.908,40.0,0.577,25.803,0,0
4,245657555.0,245657631.0,18.973,72.811,18.974,72.811,121.19,2024-01-01 04:00:00,4,0,17.479,40.0,0.563,24.960,0,0


In [23]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import os
import time

# ==============================
# CONFIG
# ==============================
OUTPUT_CSV = "synthetic_full_vasai.csv"
BATCH_SIZE = 2000
DAYS = 7
DIST = 6000

# ==============================
# 1. DOWNLOAD GRAPH
# ==============================
print("\n[STEP 1] Downloading Mumbai...\n")

lats = np.arange(19.00, 19.45, 0.05)
lons = np.arange(72.80, 73.10, 0.05)

graphs = []
tile_count = 0

for lat in lats:
    for lon in lons:

        print(f"[TILE {tile_count}] ({lat:.3f}, {lon:.3f})")

        for attempt in range(3):
            try:
                g = ox.graph_from_point(
                    (lat, lon),
                    dist=DIST,
                    network_type="drive"
                )
                graphs.append(g)
                print("   ✅ Done")
                break
            except:
                print(f"   ❌ Retry {attempt+1}")
                time.sleep(2)

        tile_count += 1

print("\n[STEP 1 DONE] Merging...\n")

G = nx.compose_all(graphs)

# ==============================
# 2. EXTRACT FULL DATA
# ==============================

nodes, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index()

# Add node coordinates
nodes = nodes[["x", "y"]]  # lon, lat

edges["u_lon"] = edges["u"].map(nodes["x"])
edges["u_lat"] = edges["u"].map(nodes["y"])
edges["v_lon"] = edges["v"].map(nodes["x"])
edges["v_lat"] = edges["v"].map(nodes["y"])

# Keep ALL useful columns (no restriction)
print("Available edge columns:", edges.columns.tolist())

# ==============================
# 3. CLEAN IMPORTANT FEATURES
# ==============================

def normalize(x):
    if isinstance(x, (list, np.ndarray)):
        return x[0] if len(x) > 0 else None
    return x

def parse_speed(x):
    x = normalize(x)
    if x is None or pd.isna(x):
        return 40.0
    try:
        return float(str(x).replace(" km/h", ""))
    except:
        return 40.0

def parse_lanes(x):
    x = normalize(x)
    if x is None or pd.isna(x):
        return 2
    try:
        return int(float(x))
    except:
        return 2

edges["maxspeed"] = edges["maxspeed"].apply(parse_speed)
edges["lanes"] = edges["lanes"].apply(parse_lanes)

edges["free_flow_speed"] = edges["maxspeed"]

road_weight = {
    "motorway": 1.0,
    "trunk": 0.9,
    "primary": 0.8,
    "secondary": 0.7,
    "tertiary": 0.6,
    "residential": 0.5
}

edges["road_weight"] = edges["highway"].apply(
    lambda x: road_weight.get(normalize(x), 0.5)
)

# ==============================
# 4. TIME VECTOR
# ==============================

timestamps = pd.date_range(
    start="2024-01-01",
    periods=24 * DAYS,
    freq="1h"
)

hours = timestamps.hour.values
days = timestamps.weekday.values
T = len(timestamps)

peak = np.where(((hours >= 8) & (hours <= 11)) | ((hours >= 17) & (hours <= 21)), 0.6, 1.0)
weekend = np.where(days >= 5, 0.8, 1.0)
time_factor = (peak * weekend)[None, :]

print(f"Time steps: {T}")

# ==============================
# 5. GENERATE DATA
# ==============================

if os.path.exists(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)

edge_features = edges[[
    "u", "v", "length", "free_flow_speed", "road_weight",
    "u_lat", "u_lon", "v_lat", "v_lon"
]].to_numpy()

for i in range(0, len(edge_features), BATCH_SIZE):

    print(f"\n🚀 Batch {i} → {min(i+BATCH_SIZE, len(edge_features))}")

    batch = edge_features[i:i+BATCH_SIZE]

    u = batch[:, 0]
    v = batch[:, 1]
    length = batch[:, 2][:, None]
    free_speed = batch[:, 3][:, None]
    road_w = batch[:, 4][:, None]

    u_lat = batch[:, 5]
    u_lon = batch[:, 6]
    v_lat = batch[:, 7]
    v_lon = batch[:, 8]

    N = len(batch)

    rain = np.random.choice([0, 1], size=(N, T), p=[0.8, 0.2])
    incident = np.random.choice([0, 1], size=(N, T), p=[0.95, 0.05])

    weather_factor = np.where(rain == 1, 0.7, 1.0)
    incident_factor = np.where(incident == 1, 0.5, 1.0)

    signal_factor = np.where(road_w < 0.7, 0.85, 1.0)
    noise = np.random.normal(1.0, 0.1, size=(N, T))

    speed = (
        free_speed
        * time_factor
        * road_w
        * weather_factor
        * incident_factor
        * signal_factor
        * noise
    )

    speed = np.clip(speed, 5, free_speed)

    congestion = 1 - (speed / free_speed)
    travel_time = length / (speed * 1000 / 3600)

    df = pd.DataFrame({
        "u": np.repeat(u, T),
        "v": np.repeat(v, T),
        "u_lat": np.repeat(u_lat, T),
        "u_lon": np.repeat(u_lon, T),
        "v_lat": np.repeat(v_lat, T),
        "v_lon": np.repeat(v_lon, T),
        "length_m": np.repeat(length.flatten(), T),
        "timestamp": np.tile(timestamps, N),
        "hour": np.tile(hours, N),
        "day_of_week": np.tile(days, N),
        "current_speed": speed.flatten(),
        "free_flow_speed": np.repeat(free_speed.flatten(), T),
        "congestion": congestion.flatten(),
        "travel_time_sec": travel_time.flatten(),
        "rain": rain.flatten(),
        "incident": incident.flatten()
    })

    df.to_csv(
        OUTPUT_CSV,
        mode='a',
        header=(i == 0),
        index=False,
        float_format="%.3f"
    )

    print("   💾 Saved")

print("\n🎉 DONE → synthetic_full.csv\n")


[STEP 1] Downloading Mumbai...

[TILE 0] (19.000, 72.800)
   ✅ Done
[TILE 1] (19.000, 72.850)
   ✅ Done
[TILE 2] (19.000, 72.900)
   ✅ Done
[TILE 3] (19.000, 72.950)
   ✅ Done
[TILE 4] (19.000, 73.000)
   ✅ Done
[TILE 5] (19.000, 73.050)
   ✅ Done
[TILE 6] (19.050, 72.800)
   ✅ Done
[TILE 7] (19.050, 72.850)
   ✅ Done
[TILE 8] (19.050, 72.900)
   ✅ Done
[TILE 9] (19.050, 72.950)
   ✅ Done
[TILE 10] (19.050, 73.000)
   ✅ Done
[TILE 11] (19.050, 73.050)
   ✅ Done
[TILE 12] (19.100, 72.800)
   ✅ Done
[TILE 13] (19.100, 72.850)
   ✅ Done
[TILE 14] (19.100, 72.900)
   ✅ Done
[TILE 15] (19.100, 72.950)
   ✅ Done
[TILE 16] (19.100, 73.000)
   ✅ Done
[TILE 17] (19.100, 73.050)
   ✅ Done
[TILE 18] (19.150, 72.800)
   ✅ Done
[TILE 19] (19.150, 72.850)
   ✅ Done
[TILE 20] (19.150, 72.900)
   ✅ Done
[TILE 21] (19.150, 72.950)
   ✅ Done
[TILE 22] (19.150, 73.000)
   ✅ Done
[TILE 23] (19.150, 73.050)
   ✅ Done
[TILE 24] (19.200, 72.800)
   ✅ Done
[TILE 25] (19.200, 72.850)
   ✅ Done
[TILE 26] (19.2

In [24]:
data_3=pd.read_csv('synthetic_full_vasai.csv')

In [25]:
len(data_3)

24176712

In [27]:
data_3.head()

,u,v,u_lat,u_lon,v_lat,v_lon,length_m,timestamp,hour,day_of_week,current_speed,free_flow_speed,congestion,travel_time_sec,rain,incident
0,245654691.0,1.034966e+09,18.946,72.832,18.947,72.833,181.076,2024-01-01 00:00:00,0,0,19.192,40.0,0.520,33.965,1,0
1,245654691.0,1.034966e+09,18.946,72.832,18.947,72.833,181.076,2024-01-01 01:00:00,1,0,26.770,40.0,0.331,24.351,0,0
2,245654691.0,1.034966e+09,18.946,72.832,18.947,72.833,181.076,2024-01-01 02:00:00,2,0,22.672,40.0,0.433,28.752,1,0
3,245654691.0,1.034966e+09,18.946,72.832,18.947,72.833,181.076,2024-01-01 03:00:00,3,0,21.975,40.0,0.451,29.665,1,0
4,245654691.0,1.034966e+09,18.946,72.832,18.947,72.833,181.076,2024-01-01 04:00:00,4,0,23.411,40.0,0.415,27.845,1,0
